# --- Model Orchestrator ---
Verify model initialization, layer channel shapes, and multi-head routing logic.


In [47]:
# ==========================
# Libraries & Custom Scripts
# ==========================
import os
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import torch


In [48]:
# =============================
# Create Workspace directory
# =============================
BASE_DIR = Path(os.getcwd()).parent

# Connect to custom-defined modules
sys.path.append(str(BASE_DIR))

# Auto-reload scripts if any changes
%reload_ext autoreload
%autoreload 2

print("Workspace successfully configured.")

Workspace successfully configured.


In [49]:
# =========================
# Import custom Modules
# =========================
from src.models.config import MultiTaskModelConfig
from src.models.network import MultiTaskYOLO
from src.models.loss import MultiTaskUncertaintyLoss

In [50]:
# ======================================
# Test Multi-Task Network Architecture
# ======================================
print("⚙️ Initializing Multi-Task Network Model Architecture...")
config = MultiTaskModelConfig()
# Notice the parentheses () added to config.weights_path()
weights_file = config.weights_path() if callable(config.weights_path) else config.weights_path

# Verify file existence before initialization
if os.path.exists(config.weights_path):
    print("✅ Weights file located successfully!")
else:
    print(f"⚠️ Warning: File not found at {config.weights_path}. Ultralytics will auto-download it there.")

model = MultiTaskYOLO(config)
print("✅ MultiTaskYOLO successfully instantiated with weights from isolated folder!")

# Dummy batch input simulating DataLoader outputs
dummy_images = torch.randn(8, 3, 640, 640)
dummy_task_ids = torch.tensor([0, 0, 1, 1, 2, 2, 0, 1])  # Mixed stream tasks

print("\n📋 ================= MODEL CONFIGURATION REPORT =================")
print(f"• Configured Backbone : {config.backbone_type.upper()}")
print(f"• Active Task Heads   : {list(config.heads.keys())}")
for name, cfg in config.heads.items():
    print(f"- [{name.upper()}] Classes: {cfg.num_classes} | Loss Weight: {cfg.loss_weight}")

# Test forward pass
model.eval()
with torch.no_grad():
    outputs = model(dummy_images, dummy_task_ids)

print("✅ Model forward pass completed successfully without argument mismatch errors!")

print("\n✅ SUCCESS: Network Multi-Head Routing Verification Complete!")
for task_name, res in outputs.items():
    print(f"• Head '{task_name}' processed {res['mask'].sum().item()} samples from batch.")

⚙️ Initializing Multi-Task Network Model Architecture...
⚠️ Warning: File not found at /teamspace/studios/this_studio/aiport-incident-detection-final/weights/yolov8m.pt. Ultralytics will auto-download it there.
✅ MultiTaskYOLO successfully instantiated with weights from isolated folder!

📋 ================= MODEL CONFIGURATION REPORT =================
• Configured Backbone : YOLOV8M
• Active Task Heads   : ['turnaround', 'ppe', 'fod']
- [TURNAROUND] Classes: 13 | Loss Weight: 1.0
- [PPE] Classes: 2 | Loss Weight: 1.2
- [FOD] Classes: 1 | Loss Weight: 1.5
✅ Model forward pass completed successfully without argument mismatch errors!

✅ SUCCESS: Network Multi-Head Routing Verification Complete!
• Head 'turnaround' processed 3 samples from batch.
• Head 'ppe' processed 3 samples from batch.
• Head 'fod' processed 2 samples from batch.


In [51]:
# ================================
# Verify: Loss computation,
# Uncertainty parameter gradients,
# & logging dict keys
# ================================
loss_fn = MultiTaskUncertaintyLoss(model, config)

print("✅ SUCCESS: Multi-Task Uncertainty Loss Module initialized cleanly!")
print("\n📋 Learnable Task Uncertainty Parameters:")
for task, param in loss_fn.log_vars.items():
    print(f"• Head '{task}': log_var = {param.item():.4f} | initial sigma = {torch.exp(0.5 * param).item():.4f}")

print("\n✅ Multi-Task Uncertainty Loss Module initialized successfully!")

✅ SUCCESS: Multi-Task Uncertainty Loss Module initialized cleanly!

📋 Learnable Task Uncertainty Parameters:
• Head 'fod': log_var = 0.0000 | initial sigma = 1.0000
• Head 'ppe': log_var = 0.0000 | initial sigma = 1.0000
• Head 'turnaround': log_var = 0.0000 | initial sigma = 1.0000

✅ Multi-Task Uncertainty Loss Module initialized successfully!
